In [ ]:
import os
import glob
import time
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# ---- project paths -------------------------------------------------------
RAW_DIR = os.path.join("data", "raw")
CSV_DIR = os.path.join("data", "csv")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

print("Raw parquet folder :", os.path.abspath(RAW_DIR))
print("Output csv folder  :", os.path.abspath(CSV_DIR))


In [ ]:
parquet_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.parquet")))
print(f"Found {len(parquet_files)} parquet file(s):")
for f in parquet_files:
    print(" -", f)

assert len(parquet_files) > 0, (
    "No parquet files found in data/raw/. Either run the download cell above, "
    "or manually copy your *.parquet file(s) into data/raw/ and re-run this cell."
)


In [ ]:
frames = []
for f in parquet_files:
    t0 = time.time()
    df_part = pd.read_parquet(f, engine="pyarrow")
    print(f"{os.path.basename(f):40s} rows={len(df_part):>10,d}  cols={df_part.shape[1]:2d}  "
          f"({time.time() - t0:.1f}s)")
    frames.append(df_part)

df_raw = pd.concat(frames, ignore_index=True)
del frames
print("\nCombined raw shape:", df_raw.shape)
df_raw.head()


In [ ]:
print("Columns in raw data:")
print(list(df_raw.columns))
print("\nDtypes:")
print(df_raw.dtypes)
print("\nShape:", df_raw.shape)
print("\nNull counts:")
print(df_raw.isnull().sum())


In [ ]:
df_raw.describe(include="all").T


In [ ]:
ATTRIBUTE_DOCS = {
    "VendorID": "Code indicating the TPEP provider that supplied the record (1 = Creative Mobile Technologies, 2 = VeriFone Inc.)",
    "tpep_pickup_datetime": "Date and time when the meter was engaged (trip start)",
    "tpep_dropoff_datetime": "Date and time when the meter was disengaged (trip end)",
    "passenger_count": "Number of passengers in the vehicle (driver-entered value)",
    "trip_distance": "Elapsed trip distance in miles reported by the taximeter",
    "pickup_longitude": "Longitude where the meter was engaged",
    "pickup_latitude": "Latitude where the meter was engaged",
    "dropoff_longitude": "Longitude where the meter was disengaged",
    "dropoff_latitude": "Latitude where the meter was disengaged",
    "PULocationID": "TLC Taxi Zone in which the taximeter was engaged (newer schema)",
    "DOLocationID": "TLC Taxi Zone in which the taximeter was disengaged (newer schema)",
    "RateCodeID": "Final rate code in effect at trip end (1=Standard,2=JFK,3=Newark,4=Nassau/Westchester,5=Negotiated,6=Group ride)",
    "store_and_fwd_flag": "Y if trip record was held in vehicle memory before sending to vendor (no connection), else N",
    "payment_type": "Numeric code for how the passenger paid (1=Credit card,2=Cash,3=No charge,4=Dispute,5=Unknown,6=Voided trip)",
    "fare_amount": "Time-and-distance fare calculated by the meter (USD)",
    "extra": "Miscellaneous extras/surcharges (rush hour, overnight, etc., USD)",
    "mta_tax": "$0.50 MTA tax automatically triggered by the metered rate in use",
    "tip_amount": "Tip amount (auto-populated for credit-card payments)",
    "tolls_amount": "Total amount of tolls paid on the trip",
    "improvement_surcharge": "$0.30 improvement surcharge assessed on hailed trips",
    "total_amount": "Total amount charged to passengers (does not include cash tips)",
}

present_cols = [c for c in df_raw.columns if c in ATTRIBUTE_DOCS]
dict_df = pd.DataFrame(
    {"attribute": present_cols,
     "description": [ATTRIBUTE_DOCS[c] for c in present_cols]}
)
dict_df


In [ ]:
df = df_raw.copy()
start_rows = len(df)
report = []  # (rule, rows_removed, rows_remaining)

def log_step(name, before):
    after = len(df)
    report.append((name, before - after, after))
    print(f"{name:45s}  removed={before - after:>10,d}   remaining={after:>10,d}")


In [ ]:
# --- 5.1 Drop exact duplicate rows -----------------------------------------
before = len(df)
df = df.drop_duplicates()
log_step("Drop duplicate rows", before)


In [ ]:
# --- 5.2 Parse datetimes & drop rows with missing/unparseable timestamps ---
before = len(df)
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")
df = df.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime"])
log_step("Drop rows with invalid pickup/dropoff datetime", before)


In [ ]:
# --- 5.3 Trip must end after it starts, and last a realistic duration ------
before = len(df)
df["trip_duration_min"] = (
    (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60.0
)
# keep trips between 1 minute and 3 hours (180 min) — filters GPS glitches / stuck meters
df = df[(df["trip_duration_min"] >= 1) & (df["trip_duration_min"] <= 180)]
log_step("Keep trip_duration_min in [1, 180] minutes", before)


In [ ]:
# --- 5.4 Passenger count must be realistic for a yellow cab -----------------
before = len(df)
if "passenger_count" in df.columns:
    df["passenger_count"] = pd.to_numeric(df["passenger_count"], errors="coerce")
    df = df.dropna(subset=["passenger_count"])
    df["passenger_count"] = df["passenger_count"].astype(int)
    df = df[(df["passenger_count"] >= 1) & (df["passenger_count"] <= 6)]
log_step("Keep passenger_count in [1, 6]", before)


In [ ]:
# --- 5.5 Trip distance must be positive and within a sane upper bound -------
before = len(df)
df["trip_distance"] = pd.to_numeric(df["trip_distance"], errors="coerce")
df = df.dropna(subset=["trip_distance"])
df = df[(df["trip_distance"] > 0) & (df["trip_distance"] <= 100)]
log_step("Keep 0 < trip_distance <= 100 miles", before)


In [ ]:
# --- 5.6 VendorID must be one of the documented codes -----------------------
before = len(df)
if "VendorID" in df.columns:
    df = df[df["VendorID"].isin([1, 2])]
log_step("Keep VendorID in {1, 2}", before)


In [ ]:
# --- 5.7 RateCodeID must be one of the documented codes ---------------------
before = len(df)
if "RateCodeID" in df.columns:
    df["RateCodeID"] = pd.to_numeric(df["RateCodeID"], errors="coerce")
    df = df.dropna(subset=["RateCodeID"])
    df["RateCodeID"] = df["RateCodeID"].astype(int)
    df = df[df["RateCodeID"].isin([1, 2, 3, 4, 5, 6])]
log_step("Keep RateCodeID in {1..6}", before)


In [ ]:
# --- 5.8 store_and_fwd_flag must be Y or N -----------------------------------
before = len(df)
if "store_and_fwd_flag" in df.columns:
    df["store_and_fwd_flag"] = df["store_and_fwd_flag"].astype(str).str.strip().str.upper()
    df = df[df["store_and_fwd_flag"].isin(["Y", "N"])]
log_step("Keep store_and_fwd_flag in {Y, N}", before)


In [ ]:
# --- 5.9 GPS coordinates must fall within the NYC bounding box --------------
# (only applies to the older lat/long schema; skipped automatically on newer schema)
NYC_LAT_MIN, NYC_LAT_MAX = 40.40, 41.10
NYC_LON_MIN, NYC_LON_MAX = -74.35, -73.60

geo_cols = ["pickup_latitude", "pickup_longitude", "dropoff_latitude", "dropoff_longitude"]
if all(c in df.columns for c in geo_cols):
    before = len(df)
    for c in geo_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=geo_cols)
    df = df[
        df["pickup_latitude"].between(NYC_LAT_MIN, NYC_LAT_MAX)
        & df["dropoff_latitude"].between(NYC_LAT_MIN, NYC_LAT_MAX)
        & df["pickup_longitude"].between(NYC_LON_MIN, NYC_LON_MAX)
        & df["dropoff_longitude"].between(NYC_LON_MIN, NYC_LON_MAX)
        # (0, 0) is a classic "GPS failed to acquire" placeholder
        & ~((df["pickup_latitude"] == 0) & (df["pickup_longitude"] == 0))
        & ~((df["dropoff_latitude"] == 0) & (df["dropoff_longitude"] == 0))
    ]
    log_step("Keep pickup/dropoff coordinates inside NYC bounding box", before)
else:
    print("Lat/long columns not present (newer schema) — skipping geo filter,"
          " will validate PULocationID/DOLocationID instead if present.")
    if "PULocationID" in df.columns and "DOLocationID" in df.columns:
        before = len(df)
        df = df.dropna(subset=["PULocationID", "DOLocationID"])
        log_step("Drop rows with missing PULocationID/DOLocationID", before)


In [ ]:
# --- 5.10 Fare / money fields must be non-negative (if present) -------------
money_cols = [c for c in ["fare_amount", "extra", "mta_tax", "tip_amount",
                           "tolls_amount", "improvement_surcharge", "total_amount"]
              if c in df.columns]
if money_cols:
    before = len(df)
    for c in money_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=money_cols)
    # negative fares/totals are refunds or data errors for this analysis
    df = df[(df[money_cols] >= 0).all(axis=1)]
    # a fare of 0 with a real trip distance is also implausible — light sanity cap
    if "total_amount" in df.columns:
        df = df[df["total_amount"] <= 500]
    log_step("Keep non-negative fare/money fields (total_amount <= 500)", before)


In [ ]:
print(f"\nTOTAL removed: {start_rows - len(df):,d} of {start_rows:,d} rows "
      f"({(start_rows - len(df)) / start_rows:.1%})")
print(f"Remaining clean rows: {len(df):,d}")

pd.DataFrame(report, columns=["cleaning_step", "rows_removed", "rows_remaining"])


In [ ]:
df["pickup_date"] = df["tpep_pickup_datetime"].dt.date.astype(str)
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day"] = df["tpep_pickup_datetime"].dt.day
df["pickup_month"] = df["tpep_pickup_datetime"].dt.month
df["pickup_year"] = df["tpep_pickup_datetime"].dt.year
df["pickup_dayofweek"] = df["tpep_pickup_datetime"].dt.dayofweek  # 0=Monday
df["pickup_day_name"] = df["tpep_pickup_datetime"].dt.day_name()
df["is_weekend"] = df["pickup_dayofweek"].isin([5, 6]).astype(int)

# trip_duration_min already computed during cleaning; round for readability
df["trip_duration_min"] = df["trip_duration_min"].round(2)

# average speed (mph) — guard against divide-by-zero (already excluded duration==0)
df["avg_speed_mph"] = (df["trip_distance"] / (df["trip_duration_min"] / 60.0)).round(2)
# drop the small number of physically-impossible speeds (sensor noise)
before = len(df)
df = df[(df["avg_speed_mph"] > 0) & (df["avg_speed_mph"] <= 80)]
print(f"Removed {before - len(df):,d} rows with implausible avg_speed_mph (>80 mph)")

# simple fare-per-mile if money columns exist — useful for a Hive/MapReduce metric
if "total_amount" in df.columns:
    df["fare_per_mile"] = (df["total_amount"] / df["trip_distance"]).round(2)

df.head()


In [ ]:
print("Final cleaned shape:", df.shape)
df.dtypes


In [ ]:
CLEANED_PATH = os.path.join(CSV_DIR, "yellow_tripdata_cleaned.csv")
df.to_csv(CLEANED_PATH, index=False)
print(f"Wrote {len(df):,d} rows -> {CLEANED_PATH} "
      f"({os.path.getsize(CLEANED_PATH) / 1e6:.1f} MB)")


In [ ]:
SAMPLE_SIZE = 50_000  # >= 5,000 required by the rubric; adjust as needed
SAMPLE_PATH = os.path.join(CSV_DIR, "yellow_tripdata_sample_for_hadoop.csv")

sample_n = min(SAMPLE_SIZE, len(df))
df_sample = df.sample(n=sample_n, random_state=42).sort_values("tpep_pickup_datetime")
df_sample.to_csv(SAMPLE_PATH, index=False)
print(f"Wrote {len(df_sample):,d} rows -> {SAMPLE_PATH} "
      f"({os.path.getsize(SAMPLE_PATH) / 1e6:.1f} MB)")


In [ ]:
DICT_PATH = os.path.join(CSV_DIR, "data_dictionary.csv")
dict_df.to_csv(DICT_PATH, index=False)
print(f"Wrote data dictionary -> {DICT_PATH}")


In [ ]:
print("\nFiles now in data/csv/:")
for f in sorted(glob.glob(os.path.join(CSV_DIR, "*.csv"))):
    print(f" - {f}  ({os.path.getsize(f) / 1e6:.2f} MB)")


In [ ]:
check = pd.read_csv(SAMPLE_PATH)
print("Re-read shape:", check.shape)
assert check.shape == df_sample.shape, "Row/column mismatch after CSV round-trip!"
assert check.isnull().sum().sum() == 0, "Unexpected NaNs introduced by CSV round-trip!"
print("CSV round-trip OK — safe to hand off to HDFS / Hive / MapReduce.")
check.head()
